In [9]:
import sqlite3
import random
import time
from datetime import date, timedelta
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA temp_store = MEMORY")
conn.execute("PRAGMA cache_size = -200000")

def sql(query, params=None, limit=20):
    params = params or ()
    df = pd.read_sql_query(query, conn, params=params)
    return df if limit is None else df.head(limit)

def explain(query):
    rows = conn.execute("EXPLAIN QUERY PLAN " + query).fetchall()
    return pd.DataFrame(rows, columns=["id", "parent", "notused", "detail"])

def time_sql(query, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        conn.execute(query).fetchall()
        times.append(time.perf_counter() - t0)
    times.sort()
    return {
        "median_ms": round(times[len(times)//2] * 1000, 2),
        "all_ms": [round(x * 1000, 2) for x in times]
    }

def drop_non_pk_indexes():
    indexes = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='index'
          AND name NOT LIKE 'sqlite_autoindex%'
    """).fetchall()
    for (name,) in indexes:
        conn.execute(f"DROP INDEX IF EXISTS {name}")
    conn.commit()

conn.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    user_id      INTEGER PRIMARY KEY,
    signup_date  TEXT NOT NULL,
    channel      TEXT NOT NULL,
    city         TEXT NOT NULL
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    category     TEXT NOT NULL,
    brand        TEXT NOT NULL,
    base_price   REAL NOT NULL
);

CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    user_id      INTEGER NOT NULL,
    order_date   TEXT NOT NULL,
    status       TEXT NOT NULL,
    promo_code   TEXT,
    FOREIGN KEY(user_id) REFERENCES users(user_id)
);

CREATE TABLE order_items (
    item_id      INTEGER PRIMARY KEY,
    order_id     INTEGER NOT NULL,
    product_id   INTEGER NOT NULL,
    quantity     INTEGER NOT NULL,
    price        REAL NOT NULL,
    FOREIGN KEY(order_id) REFERENCES orders(order_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
);
""")

channels = ["organic", "ads", "referral", "partners"]
cities = ["Moscow", "SPb", "Kazan", "Novosibirsk", "Ekaterinburg"]
start = date(2024, 1, 1)

users = []
N_USERS = 12_000
for uid in range(1, N_USERS + 1):
    signup = start + timedelta(days=random.randint(0, 240))
    channel = random.choices(channels, weights=[40, 30, 18, 12])[0]
    city = random.choices(cities, weights=[42, 24, 12, 12, 10])[0]
    users.append((uid, signup.isoformat(), channel, city))
conn.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", users)

categories = ["Electronics", "Home", "Beauty", "Sport", "Books", "Kids"]
products = []
N_PRODUCTS = 600
for pid in range(1, N_PRODUCTS + 1):
    category = categories[(pid - 1) % len(categories)]
    brand = f"brand_{1 + (pid % 35)}"
    price = round(random.uniform(300, 15000), 2)
    products.append((pid, category, brand, price))
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products)
product_lookup = {pid: (category, price) for pid, category, brand, price in products}

orders = []
N_ORDERS = 60_000
order_start = date(2024, 2, 1)
for oid in range(1, N_ORDERS + 1):
    uid = random.randint(1, N_USERS)
    d = order_start + timedelta(days=random.randint(0, 211))
    status = random.choices(
        ["paid", "cancelled", "refunded", "created"],
        weights=[72, 12, 6, 10]
    )[0]
    promo = random.choice([None, None, None, "WELCOME", "SALE10", "VIP"])
    orders.append((oid, uid, d.isoformat(), status, promo))
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

items = []
item_id = 1
for oid, uid, d, status, promo in orders:
    n_items = random.choices([1, 2, 3, 4, 5], weights=[30, 30, 22, 12, 6])[0]
    for _ in range(n_items):
        pid = random.randint(1, N_PRODUCTS)
        _, base_price = product_lookup[pid]
        qty = random.choices([1, 2, 3], weights=[78, 18, 4])[0]
        price = round(base_price * random.uniform(0.85, 1.08), 2)
        items.append((item_id, oid, pid, qty, price))
        item_id += 1
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", items)
conn.commit()
drop_non_pk_indexes()

print("users:", conn.execute("SELECT COUNT(*) FROM users").fetchone()[0])
print("orders:", conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print("order_items:", conn.execute("SELECT COUNT(*) FROM order_items").fetchone()[0])
print("products:", conn.execute("SELECT COUNT(*) FROM products").fetchone()[0])

users: 12000
orders: 60000
order_items: 140510
products: 600


## Схема данных

- `users(user_id, signup_date, channel, city)` - пользователи
- `orders(order_id, user_id, order_date, status, promo_code)` - заказы
- `order_items(item_id, order_id, product_id, quantity, price)` - позиции заказов
- `products(product_id, category, brand, base_price)` - товары

# Часть 1. Повторение

## Задача 1. Метрики заказов по городам

За **август 2024** для каждого города посчитайте:
- общее количество заказов;
- количество оплаченных заказов;
- количество отменённых заказов (`cancelled`);
- долю оплаченных заказов среди всех заказов.

Оставьте только города, в которых за август было **не меньше 900 заказов**.  
Отсортируйте результат по `paid_rate` по убыванию.

In [10]:
q1 = """
select
    u.city,
    count(*) as all_orders,
    sum(case when o.status = 'paid' then 1 else 0 end) as paid_orders,
    sum(case when o.status = 'cancelled' then 1 else 0 end) as cancelled_orders,
    round(1.0 * sum(case when o.status = 'paid' then 1 else 0 end) / count(*), 3) as paid_rate
from
    orders o
    join
    users u
        on o.user_id = u.user_id
where
    o.order_date >= '2024-08-01'
    and
    o.order_date < '2024-09-01'
group by
    u.city
having
    count(*) >= 900
order by
    paid_rate desc
"""

sql(q1)

,city,all_orders,paid_orders,cancelled_orders,paid_rate
0,Novosibirsk,993,727,119,0.732
1,SPb,2030,1466,244,0.722
2,Moscow,3542,2507,441,0.708
3,Kazan,1045,730,128,0.699


## Задача 2. Топ-2 бренда внутри каждой категории

Для оплаченных заказов за **август 2024** посчитайте выручку каждого бренда внутри каждой категории.

Затем для каждой категории оставьте **два бренда с максимальной выручкой**.  
Если выручка двух брендов совпала, они должны получить одинаковый ранг.

В результате нужны: `category`, `brand`, `revenue`, `rank_in_category`.

In [11]:
# row_number
# val pos
# 1   1
# 1   2
# 4   3

# rank
# 1 1
# 1 1
# 4 3

# dense_rank
# 1 1
# 1 1
# 4 2

In [12]:
q2 = """
with brand_revenue as (
    select
        p.category,
        p.brand,
        (oi.quantity * oi.price) as revenue
    from
        orders o
        join
        order_items oi
            on o.order_id = oi.order_id
        join
        products p
            on oi.product_id = p.product_id
    where
        o.status = 'paid'
        and
        o.order_date >= '2024-08-01'
        and
        o.order_date < '2024-09-01'
    group by
        p.category,
        p.brand
),
ranked as (
    select
        *,
        dense_rank() over (
            partition by category
            order by revenue desc
        ) as rn
    from
        brand_revenue
)
select
    category,
    brand,
    round(revenue, 2) as revenue,
    rn
from
    ranked
where
    rn <= 2
order by
    category,
    rn,
    brand
"""
sql(q2)

,category,brand,revenue,rn
0,Beauty,brand_24,40516.44,1
1,Beauty,brand_4,34261.08,2
2,Books,brand_15,32698.35,1
3,Books,brand_29,29863.84,2
4,Electronics,brand_31,29442.88,1
5,Electronics,brand_27,15918.06,2
6,Home,brand_5,24643.80,1
7,Home,brand_28,16300.32,2
8,Kids,brand_21,29359.88,1
9,Kids,brand_24,28540.52,2


## Задача 3. Скользящее среднее по дням

Для каждого дня посчитайте количество **оплаченных заказов**.

Добавьте колонку с **7-дневным скользящим средним** количества оплаченных заказов: текущий день + 6 предыдущих строк.

В результате нужны: `dt`, `paid_orders`, `ma7_paid_orders`.

In [13]:
q3 = """
with daily as (
    select
        order_date as dt,
        count(*) as paid_orders
    from
        orders
    where
        status = 'paid'
    group by
        order_date
)
select
    dt,
    paid_orders,
    round(avg(paid_orders) over ( -- партиция в функции avg
        order by dt
        rows between 6 preceding and current row
    ), 2) as ma7_paid_orders
from
    daily
order by
    dt
"""

sql(q3)

,dt,paid_orders,ma7_paid_orders
0,2024-02-01,204,204.00
1,2024-02-02,196,200.00
2,2024-02-03,217,205.67
3,2024-02-04,207,206.00
4,2024-02-05,213,207.40
5,2024-02-06,217,209.00
6,2024-02-07,205,208.43
7,2024-02-08,216,210.14
8,2024-02-09,210,212.14
9,2024-02-10,208,210.86


## Задача 4. Реактивации пользователей

Будем считать заказ **реактивацией**, если пользователь сделал оплаченный заказ спустя **45 дней или больше** после своего предыдущего оплаченного заказа.

Найдите такие заказы и затем по каждому каналу привлечения посчитайте:
- число заказов-реактиваций;
- число уникальных пользователей, у которых была хотя бы одна реактивация.

Отсортируйте по числу реактивированных пользователей по убыванию.

In [14]:
q4 = """
with paid as (
    select
        user_id,
        order_id,
        order_date,
        lag(order_date) over (
            partition by user_id
            order by order_date, order_id
        ) as prev_paid_date
    from
        orders
    where
        status = 'paid'
),
reactivated as (
    select
        user_id,
        order_id,
        order_date,
        prev_paid_date,
        cast(julianday(order_date) - julianday(prev_paid_date) as int) as gap
    from
        paid
    where
        prev_paid_date is not null
        and
        julianday(order_date) - julianday(prev_paid_date) >= 45
)
select
    u.channel,
    count(*) as reactivation_orders,
    count(distinct r.user_id) as reactivated_users
from
    reactivated r
    join
    users u
        on r.user_id = u.user_id
group by
    u.channel
order by
    reactivated_users desc
"""

sql(q4)

,channel,reactivation_orders,reactivated_users
0,organic,4300,3284
1,ads,3374,2602
2,referral,1910,1476
3,partners,1245,963


## Задача 5. Сегментация покупателей по выручке

Для каждого пользователя с хотя бы одним оплаченным заказом посчитайте:
- число оплаченных заказов;
- общую сумму покупок.

Разбейте пользователей на **4 квартиля по общей сумме покупок**, от самых больших трат к самым маленьким.

Для каждого квартиля покажите:
- число пользователей;
- среднюю сумму покупок;
- среднее число оплаченных заказов;
- минимальную и максимальную сумму покупок в квартиле.

In [15]:
q5 = """
with user_stats as (
    select
        o.user_id,
        count(distinct o.order_id) as paid_orders,
        sum(oi.quantity * oi.price) as revenue
    from
        orders o
        join
        order_items oi
            on o.order_id = oi.order_id
    where
        o.status = 'paid'
    group by
        o.user_id
),
segmented as (
    select
        *,
        ntile(4) over (order by revenue desc) as revenue_group
    from
        user_stats
)
select
    revenue_group,
    count(*) as total_users,
    round(avg(revenue), 2) as avg_revenue,
    round(avg(paid_orders), 2) as avg_paid_orders,
    min(revenue) as min_revenue,
    max(revenue) as max_revenue
from
    segmented
group by
    revenue_group
order by
    revenue_group
"""
sql(q5)

,revenue_group,total_users,avg_revenue,avg_paid_orders,min_revenue,max_revenue
0,1,2922,149476.75,5.64,110606.23,433865.81
1,2,2922,91704.96,4.08,74367.71,110600.37
2,3,2921,59594.16,3.16,44892.18,74348.14
3,4,2921,26886.01,1.90,581.59,44888.60


# Часть 2. Оптимизация запросов

Для каждой задачи работаем одинаково:
1. пишем исходный, неидеальный вариант;
2. смотрим `EXPLAIN QUERY PLAN` и время;
3. формулируем проблему;
4. меняем запрос и/или индекс;
5. снова смотрим план и время.

## Задача 6. VIP-заказы: функция над колонкой и индекс

Нужно посчитать количество заказов с промокодом `VIP` за **июль 2024**.

### Этап A
Напишите рабочий вариант, в котором проверка промокода сделана через `COALESCE(promo_code, '')`.
Покажите план и время.

In [16]:
drop_non_pk_indexes()
q6_bad = """
    select
        count(*)
    from
        orders
    where
        coalesce(promo_code, '') = 'VIP'
        and
        order_date >= '2024-07-01'
        and
        order_date < '2024-08-01'
"""
display(explain(q6_bad))
print(time_sql(q6_bad))

,id,parent,notused,detail
0,3,0,216,SCAN orders


{'median_ms': 3.79, 'all_ms': [3.63, 3.76, 3.79, 3.79, 3.86]}


### Этап B
Создайте составной индекс, который кажется подходящим под промокод и дату.  
Не меняя исходный SQL, снова посмотрите план и время.

In [17]:
conn.execute("""
create index idx_order_promo_date on orders(promo_code, order_date)
"""
)
conn.commit()
display(explain(q6_bad))
print(time_sql(q6_bad))

,id,parent,notused,detail
0,3,0,214,SCAN orders USING COVERING INDEX idx_order_pro...


{'median_ms': 2.49, 'all_ms': [2.45, 2.47, 2.49, 2.55, 2.66]}


### Этап C
Перепишите условие по промокоду так, чтобы база могла использовать индекс как **поиск**, а не как полный просмотр.  
Снова сравните план и время.

In [18]:
q6_good = """
select
    count(*)
from
    orders
where
    promo_code = 'VIP'
    and
    order_date >= '2024-07-01'
    and
    order_date < '2024-08-01'
"""
display(explain(q6_good))
print(time_sql(q6_good))

,id,parent,notused,detail
0,3,0,46,SEARCH orders USING COVERING INDEX idx_order_p...


{'median_ms': 0.06, 'all_ms': [0.06, 0.06, 0.06, 0.06, 0.18]}


## Задача 7. Последние VIP-заказы: ORDER BY + LIMIT

Нужно получить **15 самых последних** заказов с промокодом `VIP`: `order_id`, `user_id`, `order_date`.

### Этап A
Удалите учебные индексы, напишите запрос, покажите план и время.  
Отдельно обратите внимание, появляется ли в плане временная структура для сортировки.

In [19]:
drop_non_pk_indexes()

q7 = """
    select
        order_id,
        user_id,
        order_date
    from
        orders
    where
        promo_code = 'VIP'
    order by
        order_date DESC,
        order_id DESC
    limit
        15
"""
display(explain(q7))
print(time_sql(q7))

,id,parent,notused,detail
0,4,0,216,SCAN orders
1,19,0,0,USE TEMP B-TREE FOR ORDER BY


{'median_ms': 3.7, 'all_ms': [3.52, 3.56, 3.7, 3.71, 3.82]}


### Этап B
Создайте индекс, который одновременно помогает:
- отфильтровать `VIP`;
- получить строки уже в нужном порядке для `ORDER BY`.

Повторите `EXPLAIN` и измерение времени.

In [20]:
conn.execute("""
create index idx_orders_vip_latest on orders(promo_code, order_date desc, order_id desc)
""")
conn.commit()
display(explain(q7))
print(time_sql(q7))

,id,parent,notused,detail
0,5,0,63,SEARCH orders USING INDEX idx_orders_vip_lates...


{'median_ms': 0.03, 'all_ms': [0.03, 0.03, 0.03, 0.04, 0.34]}


## Задача 8. JOIN и селективность

Нужно посчитать количество товарных строк оплаченных заказов категории `Beauty` за **август 2024**.

### Этап A
Удалите учебные индексы, напишите запрос через `products → order_items → orders`, покажите план и время.

Посмотрите, с какой таблицы фактически начинает оптимизатор.

In [21]:
drop_non_pk_indexes()

q8 = """
    select
        count(*) as total_rows
    from
        products p
        join
        order_items oi
            on p.product_id = oi.product_id
        join
        orders o
            on oi.order_id = o.order_id
    where
        p.category = 'Beauty'
        and
        o.status = 'paid'
        and
        o.order_date >= '2024-08-01'
        and
        o.order_date < '2024-09-01'
"""
display(explain(q8))
print(time_sql(q8))

,id,parent,notused,detail
0,6,0,216,SCAN oi
1,8,0,45,SEARCH o USING INTEGER PRIMARY KEY (rowid=?)
2,17,0,45,SEARCH p USING INTEGER PRIMARY KEY (rowid=?)


{'median_ms': 16.33, 'all_ms': [16.02, 16.08, 16.33, 17.31, 18.91]}


### Этап B
Подумайте, какие два индекса позволят начать с маленького набора товаров нужной категории, а затем быстро найти связанные позиции заказов.

Создайте их, повторите план и измерение.

In [22]:
conn.execute("""
create index idx_products_category_pid on products(category, product_id)
""")

conn.execute("""
create index idx_items_product_order on order_items(product_id, order_id)
""")

display(explain(q8))
print(time_sql(q8))

,id,parent,notused,detail
0,6,0,55,SEARCH p USING COVERING INDEX idx_products_cat...
1,10,0,55,SEARCH oi USING COVERING INDEX idx_items_produ...
2,14,0,45,SEARCH o USING INTEGER PRIMARY KEY (rowid=?)


{'median_ms': 6.8, 'all_ms': [6.71, 6.73, 6.8, 7.31, 8.71]}


## Задача 9. Временные индексы, MATERIALIZE и BLOOM FILTER

Нужно для городов `Kazan` и `SPb` посчитать **среднее число оплаченных заказов на пользователя**, причём пользователи без оплаченных заказов тоже должны входить в знаменатель.

### Этап A
Сначала решите задачу через CTE `user_stats`, где заранее считается число оплаченных заказов на пользователя, а затем этот CTE делается `LEFT JOIN` к `users`.

Покажите план и время. Особое внимание: `MATERIALIZE`, `USE TEMP B-TREE`, `AUTOMATIC ... INDEX`, `BLOOM FILTER`.

In [23]:
drop_non_pk_indexes()

q9_bad = """
with user_stats as (
    select
        user_id,
        count(*) as paid_orders
    from
        orders
    where
        status = 'paid'
    group by
        user_id
)
select
    u.city,
    avg(coalesce(us.paid_orders, 0)) as avg_paid_orders
from
    users u
    left join
    user_stats us
        on u.user_id = us.user_id
where
    u.city in ('Kazan', 'SPb')
group by
    u.city
"""
display(sql(q9_bad))
display(explain(q9_bad))
print(time_sql(q9_bad))

,city,avg_paid_orders
0,Kazan,3.575434
1,SPb,3.549424


,id,parent,notused,detail
0,3,0,0,MATERIALIZE user_stats
1,10,3,216,SCAN orders
2,14,3,0,USE TEMP B-TREE FOR GROUP BY
3,56,0,216,SCAN u
4,65,0,0,BLOOM FILTER ON us (user_id=?)
5,75,0,47,SEARCH us USING AUTOMATIC COVERING INDEX (user...
6,82,0,0,USE TEMP B-TREE FOR GROUP BY


{'median_ms': 28.34, 'all_ms': [27.57, 27.88, 28.34, 28.62, 28.78]}


### Этап B
Перепишите расчёт без предварительно материализованного CTE: присоедините к пользователям только оплаченные заказы через условие в `ON`.

Сначала посмотрите план **без явных индексов**.

In [24]:
q9_good = """
select
    u.city,
    1.0 * count(o.order_id) / count(distinct u.user_id) as avg_paid_orders
from
    users u
    left join
    orders o
        on u.user_id = o.user_id and o.status = 'paid'
where
    u.city in ('Kazan', 'SPb')
group by
    u.city
"""
display(explain(q9_good))
print(time_sql(q9_good))

,id,parent,notused,detail
0,7,0,216,SCAN u
1,14,0,0,BLOOM FILTER ON o (status=? AND user_id=?)
2,27,0,53,SEARCH o USING AUTOMATIC PARTIAL COVERING INDE...
3,35,0,0,USE TEMP B-TREE FOR GROUP BY
4,82,0,0,USE TEMP B-TREE FOR count(DISTINCT)


{'median_ms': 53.25, 'all_ms': [51.99, 53.11, 53.25, 54.87, 57.64]}


### Этап C
Добавьте явные индексы под фильтр города и соединение пользователя с оплачиваемыми заказами.  
Снова покажите план, время и сравните результат с этапом A.

In [25]:
conn.execute("""
create index idx_users_city_id on users(city, user_id)
""")

conn.execute("""
create index idx_orders_user_status on orders(user_id, status)
""")
conn.commit()

display(sql(q9_good))
display(explain(q9_good))
print(time_sql(q9_good))

,city,avg_paid_orders
0,Kazan,3.575434
1,SPb,3.549424


,id,parent,notused,detail
0,7,0,65,SEARCH u USING COVERING INDEX idx_users_city_i...
1,25,0,54,SEARCH o USING COVERING INDEX idx_orders_user_...
2,70,0,0,USE TEMP B-TREE FOR count(DISTINCT)


{'median_ms': 4.19, 'all_ms': [3.8, 3.93, 4.19, 4.2, 4.74]}


## Задача 10. Коррелированный подзапрос

Для пользователей канала `partners` с `user_id <= 3000` нужно вывести число их заказов со статусом `refunded`.

### Этап A
Решите задачу через **коррелированный скалярный подзапрос** в `SELECT`.  
Удалите учебные индексы, затем покажите план и время.

In [26]:
drop_non_pk_indexes()

q10 = """
select
    u.user_id,
    (
        select
            count(*)
        from
            orders o
        where
            o.user_id = u.user_id
            and
            o.status = 'refunded'
    ) as refunded_users
from
    users u
where
    u.channel = 'partners'
    and
    u.user_id <= 3000
"""
display(explain(q10))
print(time_sql(q10))

,id,parent,notused,detail
0,2,0,196,SEARCH u USING INTEGER PRIMARY KEY (rowid<?)
1,11,0,0,CORRELATED SCALAR SUBQUERY 1
2,16,11,216,SCAN o


{'median_ms': 748.38, 'all_ms': [734.32, 746.95, 748.38, 882.53, 891.06]}


### Этап B
Не меняя бизнес-логику и саму форму запроса, добавьте индекс, который позволяет внутреннему подзапросу быстро находить строки конкретного пользователя и статуса.

Повторите план и время.

In [27]:
conn.execute("""
create index idx_orders_user_status on orders(user_id, status)
""")
conn.commit()
display(explain(q10))
print(time_sql(q10))

,id,parent,notused,detail
0,2,0,196,SEARCH u USING INTEGER PRIMARY KEY (rowid<?)
1,11,0,0,CORRELATED SCALAR SUBQUERY 1
2,16,11,54,SEARCH o USING COVERING INDEX idx_orders_user_...


{'median_ms': 0.71, 'all_ms': [0.56, 0.64, 0.71, 0.72, 0.77]}
